In [1]:
import pandas as pd
import numpy as np

from collections import defaultdict
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

In [2]:
data = pd.read_csv("pos_tags.csv")
data.head()

,sentence_id,word,tag
0,0,aa,NN
1,1,aaa,NN
2,2,aah,NN
3,3,aahed,VBN
4,4,aahing,VBG


In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 370100 entries, 0 to 370099
Data columns (total 3 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   sentence_id  370100 non-null  int64 
 1   word         370100 non-null  object
 2   tag          370100 non-null  object
dtypes: int64(1), object(2)
memory usage: 8.5+ MB


In [4]:
sentences = []

temp = []

current_sentence = data.iloc[0]["sentence_id"]

for _, row in data.iterrows():

    if row["sentence_id"] != current_sentence:
        sentences.append(temp)
        temp = []
        current_sentence = row["sentence_id"]

    temp.append((row["word"], row["tag"]))

sentences.append(temp)

In [5]:
train_sentences, test_sentences = train_test_split(
    sentences,
    test_size=0.2,
    random_state=42
)

In [6]:
words = set()

tags = set()

for sentence in train_sentences:

    for word, tag in sentence:
        words.add(word)
        tags.add(tag)

In [7]:
words = list(words)
tags = list(tags)

In [8]:
word2idx = {w:i for i,w in enumerate(words)}
tag2idx = {t:i for i,t in enumerate(tags)}

idx2tag = {i:t for t,i in tag2idx.items()}

In [9]:
initial_counts = np.ones(len(tags))

In [10]:
for sentence in train_sentences:
    first_tag = sentence[0][1]
    initial_counts[tag2idx[first_tag]] += 1

In [11]:
initial_prob = initial_counts / initial_counts.sum()

In [12]:
transition_counts = np.ones((len(tags), len(tags)))

In [13]:
for sentence in train_sentences:

    for i in range(len(sentence)-1):

        tag1 = tag2idx[sentence[i][1]]
        tag2 = tag2idx[sentence[i+1][1]]

        transition_counts[tag1][tag2] += 1

In [14]:
transition_prob = transition_counts / transition_counts.sum(axis=1, keepdims=True)

In [15]:
emission_counts = np.ones((len(tags), len(words)))

In [16]:
for sentence in train_sentences:

    for word, tag in sentence:

        emission_counts[
            tag2idx[tag],
            word2idx[word]
        ] += 1

In [17]:
emission_prob = emission_counts / emission_counts.sum(axis=1, keepdims=True)

In [18]:
log_initial = np.log(initial_prob)

log_transition = np.log(transition_prob)

log_emission = np.log(emission_prob)

In [19]:
def viterbi(sentence):

    T = len(sentence)
    N = len(tags)

    dp = np.full((N, T), -np.inf)
    backpointer = np.zeros((N, T), dtype=int)

    first_word = sentence[0]

    emission = np.zeros(N)

    if first_word in word2idx:
        emission = log_emission[:, word2idx[first_word]]

    dp[:,0] = log_initial + emission

    for t in range(1, T):

        emission = np.zeros(N)

        if sentence[t] in word2idx:
            emission = log_emission[:, word2idx[sentence[t]]]

        scores = dp[:,t-1][:,None] + log_transition

        backpointer[:,t] = np.argmax(scores, axis=0)

        dp[:,t] = np.max(scores, axis=0) + emission

    best_path = []

    last = np.argmax(dp[:,T-1])

    best_path.append(last)

    for t in range(T-1,0,-1):

        last = backpointer[last,t]

        best_path.append(last)

    best_path.reverse()

    return [idx2tag[i] for i in best_path]

In [20]:
true_tags = []
pred_tags = []

for sentence in test_sentences:

    words_only = [w for w,t in sentence]

    prediction = viterbi(words_only)

    pred_tags.extend(prediction)

    true_tags.extend([t for w,t in sentence])

In [21]:
print("Accuracy:", accuracy_score(true_tags, pred_tags))

print(classification_report(true_tags, pred_tags))

Accuracy: 0.6233450418805728


C:\Users\Harshini\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

          CC       0.00      0.00      0.00         3
          CD       0.00      0.00      0.00         1
          DT       0.00      0.00      0.00         6
          IN       0.00      0.00      0.00        12
          JJ       0.00      0.00      0.00      7847
         JJR       0.00      0.00      0.00         2
         JJS       0.00      0.00      0.00        62
          MD       0.00      0.00      0.00         3
          NN       0.62      1.00      0.77     46140
         NNS       0.00      0.00      0.00      9682
         PRP       0.00      0.00      0.00         4
        PRP$       0.00      0.00      0.00         3
          RB       0.00      0.00      0.00      3457
         RBR       0.00      0.00      0.00         1
          TO       0.00      0.00      0.00         1
          VB       0.00      0.00      0.00       346
         VBD       0.00      0.00      0.00       130
         VBG       0.00    

C:\Users\Harshini\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Harshini\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [22]:
test_sentences = [
    "I love machine learning",
    "The cat is sleeping",
    "She writes beautiful poems",
    "Python is amazing",
    "Students are studying"
]

for s in test_sentences:

    words = s.split()

    print("\nSentence:", s)

    print(list(zip(words, viterbi(words))))


Sentence: I love machine learning
[('I', 'NN'), ('love', 'WRB'), ('machine', 'NN'), ('learning', 'VBG')]

Sentence: The cat is sleeping
[('The', 'NN'), ('cat', 'NN'), ('is', 'VBZ'), ('sleeping', 'VBG')]

Sentence: She writes beautiful poems
[('She', 'NN'), ('writes', 'NNS'), ('beautiful', 'WRB'), ('poems', 'WRB')]

Sentence: Python is amazing
[('Python', 'NN'), ('is', 'VBZ'), ('amazing', 'WRB')]

Sentence: Students are studying
[('Students', 'NN'), ('are', 'VBP'), ('studying', 'VBG')]
